In [1]:
import netCDF4 as nc
import xarray as xr
import numpy as np
import cartopy
import matplotlib.pyplot as plt

import sys
import os

%load_ext autoreload
%autoreload 2
    
# Add the directory containing the package to the search path
sys.path.append(os.path.abspath("/Users/brucel/ecco/yip/ECCO-Insitu-Python"))

# Now you can use standard absolute imports
import NCEI

# import step01 as update_prof_and_tile_points_on_profiles
# import step02 as update_spatial_bin_index_on_prepared_profiles
# import step03 as update_monthly_mean_TS_clim_WOA13v2_on_prepared_profiles
# import step04 as update_sigmaTS_on_prepared_profiles
# import step05 as update_gamma_factor_on_prepared_profiles
# import step06 as update_prof_insitu_T_to_potential_T
# import step07 as update_zero_weight_points_on_prepared_profiles
# import step08 as update_remove_zero_T_S_weighted_profiles_from_MITprof
# import step09 as update_remove_extraneous_depth_levGels
# import step10 as update_decimate_profiles_subdaily_to_once_daily
import step01
import step02
import step03
import step04
import step05
import step06
import step07
import step08
import step09
import step10
from tools import MITprof_read, MITprof_write_to_nc

In [2]:
original_file = "/Users/brucel/ecco/yip/scripps_data/CTD_WOD/WOD_WO_1992_CTD_OSD.nc"

In [3]:
processed_file = "/Users/brucel/ecco/yip/sample_data/test_output_all_files/CTD_WOD/WOD_WO_1992_CTD_OSD__ncei_step_10.nc"

In [4]:
# ==========================================================================================
# ========================== START OF NEED PATHS/ PAREMS ===================================
# ==========================================================================================

# Needed paths:
# Set grid_dir
grid_dir = '/Users/brucel/ecco/yip/sample_data/ecco-insitu/sweet_gdrive/grid_llc90'

# Path to dir containing llc090_sphere_point_n_10242_ids.bin and llc090_sphere_point_n_02562_ids.bin
sphere_dir = '/Users/brucel/ecco/yip/sample_data/ecco-insitu/sweet_gdrive/grid_llc90/sphere_point_distribution'

# Path to WOA13_v2_TS_clim_merged_with_potential_T.nc
clim_dir = '/Users/brucel/ecco/yip/sample_data/ecco-insitu/sweet_gdrive/TS_Climatology'
#clim_dir = '/Users/brucel/ecco/yip/sample_data/ecco-insitu/sweet_gdrive/TS_Climatology/WOA13_v2_TS_clim_merged_with_potential_T'

# Path to Salt_sigma_smoothed_method_02_masked_merged_capped_extrapolated.bin and Theta_sigma_smoothed_method_02_masked_merged_capped_extrapolated.bin
CTD_TS_bin = '/Users/brucel/ecco/yip/sample_data/ecco-insitu/sweet_gdrive/CTD_sigma_TS'

# Step 1: update_prof_and_tile_points_on_profiles
llcN = 90                       # Which grid to use, 90 or 270
wet_or_all = 1                  # 0 = interpolated to nearest wet point, 1 = interpolated all points, regardless of wet or dry

# Step 4: update_sigmaTS_on_prepared_profiles parems
respect_existing_zero_weights = 0   # 0 = no, 1 = yes
new_S_floor = 0.005                 # set this to zero if S_floor is unused
new_T_floor = 0                     # set this to zero if T_floor is unused

# Step 5: update_gamma_factor_on_prepared_profiles parems
apply_gamma_factor = 1          #   0 = remove gamma factor from sigma, 1 = apply gamma to sigma
                                #   gamma factor is factor 1/sqrt(alpha), where alpha = area/max(area) of the grid cell area in which this profile is found.

# Step 6: update_prof_insitu_T_to_potential_T parems
replace_missing_S_with_clim_S = 1   # 1 = replace, 0 = do not replace

# Step 7: update_zero_weight_points_on_prepared_profiles
# Various parems inside of if block pretaining to 'adjust' on lines 142 - 161 within script

# Step 10: update_decimate_profiles_subdaily_to_once_daily
distance_tolerance = 5e3        # radius within which profiles are considered to be at the same location [in meters]
closest_time = 120000           # HHMMSS: if there is more than one profile per day in a location, choose the one
                                # that is closest in time to 'closest time' default is noon
method = 1                      # method 0 or 1

In [5]:
#MITprofs = NCEI.MITprof_read(original_file,0)

largest_numbered_step_to_run = 10

#MITprofs = MITprof_read(original_file,0)
MITprofs = MITprof_read(original_file,largest_numbered_step_to_run)

In [6]:
prof_vars = list(MITprofs)

In [7]:
# bucket to hold my new, beautiful xarray data arrays 
new_dataarrays = dict()

prof_vars_dims = dict()
# find the number of depth levels
num_k = len(MITprofs['prof_depth'])
# find the number of profiles
num_profs = len(MITprofs['prof_lat'])

# loop through the different variables in the MITprofs structure
for pv in prof_vars:
    #print(MITprofs[pv].shape)
    field_shape = MITprofs[pv].shape 
    ndims = len(field_shape)
    if ndims == 1:
        if field_shape[0] == num_profs:
            #print(f'{pv} is 1D and len={num_profs}')
            new_dataarrays[pv] = xr.DataArray(MITprofs[pv], dims='iPROF', name=pv)
        elif field_shape[0] == num_k:
            #print(f'{pv} is 1D and len={num_k}')
            new_dataarrays[pv] = xr.DataArray(MITprofs[pv], dims='iDEPTH', name=pv)
    elif ndims== 2:
        #print(f'{pv} is 2D and shape is {field_shape}')
        if field_shape[0] == num_profs and field_shape[1] == num_k:
            new_dataarrays[pv] = xr.DataArray(MITprofs[pv], dims=['iPROF', 'iDEPTH'], name=pv)
        else:
            print('====== fail fail fail fail fail ===  calll help lol')

    else:
        print('====== fail fail fail fail fail ===  calll help lol')
        continue

new_dataarrays
MITprof_ds = xr.merge([new_dataarrays])

# dignity has been restored
#MITprof_ds

In [8]:
#MITprof_ds

In [9]:
%%capture
#update_prof_and_tile_points_on_profiles.main(MITprof_ds, grid_dir, llcN, wet_or_all)
step01.main(MITprof_ds, grid_dir, llcN, wet_or_all)

In [10]:
%%capture
#update_spatial_bin_index_on_prepared_profiles.main(sphere_dir, MITprof_ds, grid_dir)
step02.main(sphere_dir, MITprof_ds, grid_dir)

In [11]:
%%capture
#update_monthly_mean_TS_clim_WOA13v2_on_prepared_profiles.main(clim_dir, MITprof_ds)
step03.main(clim_dir, MITprof_ds)

In [12]:
%%capture
step04.main(MITprof_ds, grid_dir, CTD_TS_bin, respect_existing_zero_weights, new_S_floor, new_T_floor)

step 04: update_sigmaTS_on_prepared_profiles
original (line 1) vs closest (line 2) x,y,z
[0.42298782] [-0.36126573] [0.83100444]
0.42836661633533263 -0.35944226977563704 0.8290375725550417
original (line 1) vs closest (line 2) lat lon
56 -40
[56.202057] [-40.5]
original (line 1) vs closest (line 2) x,y,z
[0.51616685] [0.08640665] [0.85211835]
0.4924038765061041 0.08682408883346518 0.8660254037844386
original (line 1) vs closest (line 2) lat lon
60 10
[58.442825] [9.503233]
original (line 1) vs closest (line 2) x,y,z
[0.43807467] [-0.01912675] [-0.89873508]
0.43899160975441764 -0.01593597856961881 -0.8983498267113172
original (line 1) vs closest (line 2) lat lon
-63.942 -2.079
[-63.992294] [-2.5]
original (line 1) vs closest (line 2) x,y,z
[0.19626759] [0.33319626] [-0.92220349]
0.17918397477265022 0.3103557482083701 -0.9335804264972017
original (line 1) vs closest (line 2) lat lon
-69 60
[-67.250374] [59.5]
STEP 4: not respecting the zero weights of the original profiles


In [13]:
#MITprof_ds['prof_Tweight']

In [14]:
%%capture
#NoTe:  THIS STEP ERASES ENTIRE ROWS OF PROFILE DATA (IE ALL VALUES FOR ALL TIME AT A GIVEN DEPTH INDEX) IF ANY 
#       DATA AT ANY TIME IS BAD/NEGATIVE.  is this intended?  interspersed nans makes the data hard to assimilate?
step05.main(MITprof_ds, grid_dir, apply_gamma_factor, llcN)

In [15]:
%%capture
step06.main(MITprof_ds, replace_missing_S_with_clim_S)


In [18]:
%%capture
step07.main('adjust', MITprof_ds) 

step07: update_zero_weight_points_on_prepared_profiles
START Nonzero T weights 0
START Nonzero S weights 2157151
num profs:      64928
num nonzero T :          0
num nonzero S :      58205
num nonzero TS:      58205

Criteria : 1 T or S weight already zero

------------------------------------

Nonzero T weights NOW/ORIG 0/0
Nonzero S weights NOW/ORIG 2157151/2157151


num profs(n/o):      64928
num nonzero T :          0          0
num nonzero S :      58205      58205
num nonzero TS:          0      58205


Criteria : 2 nonzero prof T or S flag

------------------------------------

Nonzero T weights NOW/ORIG 0/0
Nonzero S weights NOW/ORIG 2154913/2157151


num profs(n/o):      64928
num nonzero T :          0          0
num nonzero S :      58148      58205
num nonzero TS:          0      58205


Criteria : 3 missing T or S

------------------------------------

Nonzero T weights NOW/ORIG 0/0
Nonzero S weights NOW/ORIG 2154913/2157151


num profs(n/o):      64928
num nonzero T :    

In [22]:
step08.main(MITprof_ds)

step08: update_remove_zero_T_S_weighted_profiles_from_MITprof
	num profs: 64928 
	num nonzero T: 0 
	num nonzero S: 19834 
	num nonzero TS: 19834
	# profs to nix: 45094
	Total T weight: <xarray.DataArray 'prof_Tweight' ()> Size: 8B
array(0.)
	Total S weight: <xarray.DataArray 'prof_Sweight' ()> Size: 8B
array(1.82053954e+08)
subsetting some profiles
including all depths


ValueError: replacement data must match the Variable's shape. replacement data has shape (19834,); Variable has shape (64928,)

In [ ]:
#import matplotlib.pyplot as plt

In [ ]:
#import cartopy.crs as ccrs
#import matplotlib.pyplot as plt

#ax = plt.axes(projection=ccrs.PlateCarree())
#ax.coastlines()

#plt.plot(MITprof_ds['prof_lon'], MITprof_ds['prof_lat'], 'k.', transform=ccrs.PlateCarree())
#plt.show()

In [ ]:
#MF.to_netcdf('X.nc')

In [ ]:
MITprof_ds['prof_Tweight']

In [ ]:
a = MITprof_ds['prof_Tweight']